## Analisis Exploratorio de Datos (EDA)

El análisis exploratorio de datos (EDA) es un proceso crucial en el análisis de datos que implica examinar y resumir las características principales de un conjunto de datos. El objetivo del EDA es comprender la estructura, las tendencias y las relaciones dentro de los datos antes de aplicar modelos estadísticos o de aprendizaje automático. A continuación, se presentan algunos pasos comunes en el proceso de EDA:

In [ ]:
# Paquetes necesarios para el análisis exploratorio de datos #
import pandas as pd

# PRESENTACIÓN: MODELO LST–ARMA

---

## INTRODUCCIÓN 


Buenos días a todos. Me complace presentar hoy el trabajo titulado **"Modelo LST–ARMA: Log-Skew-t Autoregressive Moving Average Model"**. 

Este es un trabajo de investigación desarrollado en el Instituto de Matemáticas de la Universidad de Antioquia. Mi estudiante es **Oveida Rosa Bustos Polo**, y contamos con la co-orientación del profesor **Raúl A. Morán Vásquez**.

Hoy les presentaremos una solución a un problema crucial en el análisis de series de tiempo: cómo modelar datos que son estrictamente positivos, asimétricamente distribuidos y con colas pesadas, manteniendo la interpretabilidad en la escala original.

---

## SECCIÓN 1: MOTIVACIÓN 


### ¿Por qué este modelo es necesario?

En muchos campos de aplicación—ambiental, económico y financiero—nos encontramos con un desafío recurrente: series de tiempo que son **estrictamente positivas**, presentan **datos atípicos** y **distribuciones asimétricas**.

Piensen por un momento: ¿cuál es el problema con los modelos ARMA clásicos? Estos modelos asumen que los errores siguen una distribución normal. Pero, ¿qué sucede cuando los datos reales no son normales?

**Los modelos clásicos ARMA con errores normales NO capturan adecuadamente estos comportamientos.**

### Un caso real: Calidad del Aire

Permítanme mostrar un ejemplo concreto. Aquí tenemos datos de **calidad del aire en Santa Rosa de Cabal**, monitorados por la Red de Calidad del Aire de Risaralda (IDEAM).

*[Referirse al gráfico de serie temporal]*

Observen la **serie temporal de mediciones**: hay variabilidad heterogénea, eventos atípicos puntuales, y la distribución **claramente no es simétrica**.

*[Referirse al histograma de mediciones]*

En el **histograma**, vemos una marcada asimetría positiva (sesgo hacia la derecha) y colas pesadas. Los valores no se distribuyen normalmente.

### La transformación logarítmica

¿Cómo abordamos esto? Una estrategia común es aplicar una **transformación logarítmica** para estabilizar la varianza.

*[Referirse a los gráficos de serie transformada]*

Con **log(Medición)**, la serie se vuelve más estable, pero miren el **histograma del logaritmo**: aunque mejora, aún persisten la asimetría y las colas pesadas.

### La pregunta de investigación

Esto nos lleva a la pregunta central de nuestro trabajo:

**¿Cómo formular un modelo de series de tiempo definido sobre variables positivas que, a través de una distribución flexible, permita capturar asimetría y colas pesadas en su comportamiento dinámico?**

---

## SECCIÓN 2: ESTADO DEL ARTE 


### Modelos generalizados ARMA (G-ARMA)

La literatura ofrece varias soluciones. Una de las más importantes es el **Modelo G-ARMA (Generalized ARMA)**.

La idea es extender ARMA mediante una **función de enlace g(·)** que modela la media condicional:

- El enlace conecta la escala original de los datos con un predictor lineal
- Se asume que los datos provienen de la **familia exponencial**
- Esto generaliza el ARMA normal a contextos no-gaussianos

### Modelos para variables positivas

Ahora, en la literatura se han desarrollado **modelos especializados para variables positivas**:

**1. Modelo R-ARMA (Rayleigh ARMA):**
   - Usa la distribución Rayleigh para datos positivos
   - La media logarítmica evoluciona con estructura ARMA
   - Captura variabilidad acotada y leve asimetría

**2. Modelo BXII-ARMA (Burr XII ARMA):**
   - Basado en la distribución Burr XII
   - Modeliza cuantiles condicionales en lugar de la media
   - Excelente para capturar **colas pesadas** y **asimetría**
   - Particularmente útil cuando la media no describe bien la tendencia central

### Modelos para proporciones y variables acotadas

Además, hay modelos para datos **acotados** (en intervalos como [0,1]):

**3. Modelo β-ARMA (Beta ARMA):**
   - Distribucion Beta condicional
   - La media evoluciona con estructura ARMA
   - La precisión ajusta la concentración alrededor de la media

**4. Modelo K-ARMA (Kumaraswamy ARMA):**
   - Distribucion Kumaraswamy más flexible que Beta
   - Controla asimetría y dispersión

**5. Modelo UW-ARMA (Unit-Weibull ARMA):**
   - Extiende Beta-ARMA con mayor flexibilidad
   - Permite ajustar asimetría mediante un parámetro específico

### El espacio de oportunidad

Todos estos modelos son excelentes, pero hay un **espacio de oportunidad**: 

¿Y si queremos un modelo que combine:
- **Dominio positivo** (como R-ARMA, BXII-ARMA)
- **Asimetría flexible** (como UW-ARMA)
- **Colas pesadas** (como lo hace la distribución t-Student)
- **Interpretabilidad en escala original**?

Aquí es donde entra nuestro modelo LST-ARMA.

---

## SECCIÓN 3: DISTRIBUCIÓN LOG-SKEW-T 


### Definición de la distribución

Para resolver este desafío, introducimos la **Distribución Log-Skew-t (LST)**.

Decimos que una variable aleatoria **Y > 0** tiene distribución Log-Skew-t si:

**log(Y) ~ ST(log(ξ), ω², α, ν)**

Donde:
- **ξ > 0**: parámetro de **localización**
- **ω > 0**: parámetro de **dispersión** (escala)
- **α ∈ ℝ**: parámetro de **asimetría**
- **ν > 0**: **grados de libertad** (controla el peso de las colas)

La distribución que resulta es **LST(ξ, ω², α, ν)**.

### Estructura de la función de densidad

La función de densidad de Y, para y > 0, tiene la siguiente forma:

```
f(y) = (2 / (y·ω)) · t(log(y) - log(ξ) / ω; ν) · 
       T((α·(log(y) - log(ξ)) / ω) · √((ν + 1) / (ν + δ)))
```

Donde:
- **t(·; ν)** es la densidad de la distribución t-Student
- **T(·; ν+1)** es la función acumulada de t-Student
- **δ** mide la distancia normalizada en la escala logarítmica

**¿Qué significa esto en términos prácticos?**

Esto significa que la distribución es flexible en **tres dimensiones**:
1. **Soporte positivo** (la variable Y > 0)
2. **Asimetría** (parametrizada por α)
3. **Peso de colas** (parametrizado por ν)

### Visualización de la asimetría

*[Referirse al gráfico de densidades con diferentes α]*

En este gráfico, vemos cómo **varía la forma de la densidad cuando cambia el parámetro α**:

- Cuando **α < 0**: la densidad se sesga hacia la **izquierda**
- Cuando **α = 0**: la densidad es **simétrica**
- Cuando **α > 0**: la densidad se sesga hacia la **derecha**

### Visualización del comportamiento de colas

*[Referirse al gráfico de densidades con diferentes ν]*

Aquí mostramos cómo **varía el peso de las colas con los grados de libertad ν**:

- Cuando **ν pequeño** (ej: ν = 2): **colas muy pesadas**, más probabilidad en extremos
- Cuando **ν intermedio** (ej: ν = 8): **colas moderadamente pesadas**
- Cuando **ν grande** (ej: ν = 30): **colas ligeras**, se aproxima a distribución simétrica

Esta **flexibilidad es crucial** para capturar comportamientos reales complejos.

---

## SECCIÓN 4: EL MODELO LST-ARMA 


### Especificación condicional

Ahora integramos la distribución Log-Skew-t en una **estructura dinámica ARMA**.

Sea **y_t > 0** una serie de tiempo continua. Condicionado a la información previa F_{t-1}, especificamos:

**log(y_t) | F_{t-1} ~ ST(log(ξ_t), ω², α, ν)**

Equivalentemente:

**y_t | F_{t-1} ~ LST(ξ_t, ω², α, ν)**

### Estructura de localización dinámica

El parámetro de localización **ξ_t** evoluciona según:

**log(ξ_t) = x_t^⊤ β + τ_t**

Donde:
- **x_t**: vector de **covariables exógenas**
- **β**: vector de **coeficientes de regresión**
- **τ_t**: **componente dinámica** conducida por ARMA(p,q)

### Componente ARMA

La dinámica τ_t sigue un proceso ARMA(p,q):

**τ_t = c + Σ(i=1 a p) φ_i τ_{t-i} + Σ(j=1 a q) θ_j r_{t-j}**

En forma compacta:

**Φ(B) τ_t = c + Θ(B) r_t**

Donde:
- **Φ(B)**: polinomio autorregresivo
- **Θ(B)**: polinomio de medias móviles
- **r_t**: innovaciones

### Estructura del error

El error condicional se define como:

**r_t = log(Y_t) - (x_t^⊤ β + τ_t)**

Con la transformación ξ*_t = log(Y_t) - x_t^⊤ β, obtenemos:

**ξ*_t = c + Σ φ_i ξ*_{t-i} + Σ θ_j r_{t-j} + r_t**

**Esto proporciona dos ventajas:**
1. Dinámicas interpretables en la escala logarítmica
2. Captura simultánea de tendencias, asimetría y colas pesadas

---

## SECCIÓN 5: PROPIEDADES TEÓRICAS 


### Proposición 1: Distribución del error

Una propiedad fundamental es que si log(Y_t) condicional sigue Skew-t, entonces **el error también sigue Skew-t centrado**:

Si **log(Y_t) | F_{t-1} ~ ST(log(ξ_t), ω², α, ν)**

Entonces: **r_t | F_{t-1} ~ ST(0, ω², α, ν)**

Esto significa que los residuos se distribuyen con **media cero** pero preservan la **asimetría y peso de colas**.

### Proposición 2: Cuantiles condicionales

Para cualquier nivel **0 < λ < 1**, el cuantil de orden λ de Y es:

**μ_λ = ξ · exp(ω · q_λ)**

Donde **q_λ** es el cuantil de la distribución Skew-t estándar.

En escala logarítmica:

**log(μ_λ) = log(ξ) + ω · q_λ**

**¿Por qué es importante?**

- Permite modelar **diferentes niveles** de la distribución condicional
- Proporciona **predicciones en intervalos** (cuantil bajo, mediana, cuantil alto)
- Especialmente útil en contextos de **riesgo** y **gestión de extremos**

### Proposición 3: Densidad reparametrizada por cuantil

Podemos reparametrizar la densidad en términos del **cuantil μ_t** en lugar de ξ_t:

Esto es útil porque:
- Los **cuantiles son interpretables**
- Facilita la **especificación de modelos** para cuantiles extremos
- Proporciona **mejor identificabilidad** de los parámetros

---

## SECCIÓN 6: SIMULACIONES Y RESULTADOS 


### Esquema de simulación

Para validar nuestro modelo, realizamos un **estudio de simulación Monte Carlo**:

**Parámetros generados:**
- n = 400 observaciones
- Nivel de cuantil: λ = 0.9
- 2 covariables exógenas simuladas
- Modelo AR(1) (p = 1, q = 0)

**Valores verdaderos utilizados:**

| Parámetro | Valor |
|-----------|-------|
| β₁        | 0.60  |
| β₂        | -0.40 |
| c         | -0.10 |
| φ         | 0.65  |
| ω         | 0.30  |
| α         | 1.50  |
| ν         | 8.00  |

### Propiedades de la serie simulada

*[Referirse a los gráficos de serie simulada]*

**En la escala original:**
- La trayectoria refleja **dependencia temporal** inducida por AR(1)
- El histograma muestra **asimetría positiva** clara
- Se observan **colas derechas pronunciadas**

**En la escala logarítmica:**
- La serie presenta **dinámica consistente** con el componente AR(1)
- El histograma exhibe **asimetría y colas moderadamente pesadas**
- Perfectamente consistente con nuestra especificación Skew-t

### Resultados Monte Carlo (200 réplicas)

*[Referirse a la tabla de resultados]*

Los hallazgos son **muy positivos**:

**1. Regresión:**
   - β₁ estimado: 0.5995 (verdadero: 0.60) → Sesgo casi nulo
   - β₂ estimado: -0.4001 (verdadero: -0.40) → Sesgo casi nulo
   - Baja variabilidad en ambos

**2. Dinámica AR:**
   - φ estimado: 0.6431 (verdadero: 0.65) → Sesgo: -0.0069
   - RMSE: 0.0452 → Excelente recuperación

**3. Escala:**
   - ω estimado: 0.3013 (verdadero: 0.30) → Sesgo: 0.0013
   - RMSE: 0.0333 → Muy preciso

**4. Asimetría y grados de libertad:**
   - α estimado: 1.5806 (verdadero: 1.50) → Sesgo: 0.0806
   - ν estimado: 10.6542 (verdadero: 8.00) → Sesgo: 2.6542
   - Mayor dispersión, reflejando **menor identificabilidad** en modelos Skew-t dinámicos

### Conclusiones de simulación

- ✓ **Máxima verosimilitud condicional** recupera adecuadamente parámetros de regresión y dinámica
- ✓ **Ajuste distribucional** consistente con especificación Skew-t
- ✓ **Parametrización por cuantil** estable y coherente
- ⚠ Los parámetros de colas (ν) requieren **muestras más grandes** para mayor precisión

---

## CONCLUSIONES FINALES 


### Resumen ejecutivo

Hemos presentado el **Modelo LST-ARMA**, una contribución que:

1. **Resuelve un problema real**: modelar series positivas, asimétricas con colas pesadas
2. **Combina flexibilidad**: distribución Skew-t + dinámica ARMA
3. **Mantiene interpretabilidad**: en escala original mediante transformación logarítmica
4. **Demuestra solidez**: a través de estudios Monte Carlo exhaustivos

### Aplicaciones futuras

Este modelo abre oportunidades para:
- Análisis de **calidad ambiental**
- Modelado de **rendimientos financieros**
- Predicción de **variables económicas positivas**
- Estudios de **confiabilidad y tiempos de vida**

### Gracias y apertura a preguntas

**Agradezco su atención. ¿Preguntas o comentarios?**

---

## NOTAS PARA EL PRESENTADOR

### Ritmo y énfasis

- **Motivación** (Minutos 1-4): Conectar emocionalmente con el problema real
- **Estado del arte** (Minutos 4-9): Mostrar contexto de investigación
- **Teoría** (Minutos 9-18): Equilibrio entre rigor y claridad
- **Validación** (Minutos 18-24): Evidencia empírica sólida
- **Conclusión** (Minuto 24-25): Impacto y relevancia futura

### Recomendaciones de presentación

1. **Uso de visualización**: Enfatice los gráficos de densidad y series temporales
2. **Notación matemática**: Presente ecuaciones lentamente, explique cada término
3. **Ejemplos concretos**: Vuelva al caso de calidad del aire regularmente
4. **Pausas reflexivas**: Permita que la audiencia procese conceptos complejos
5. **Lenguaje accesible**: Aunque es trabajo técnico, use analogías cuando sea posible

### Posibles preguntas de la audiencia

- ¿Cómo compararía este modelo con GARCH o ARIMAX?
- ¿Qué sucede si los datos no son estrictamente positivos?
- ¿Cuál es el costo computacional de la estimación?
- ¿Puede extenderse a multivariado?
